# Assignment-4 — Cognitive Computing

## Q1: Build Your Personalized Knowledge Base



In [1]:
roll_no = 1024160010

# Extract the last two digits for the personalized FAQ entries
last_two_digits_str = str(roll_no)[-2:]
digit1 = int(last_two_digits_str[0])
digit2 = int(last_two_digits_str[1])

print("Roll Number:", roll_no)
print("Last two digits:", digit1, digit2)


Roll Number: 1024160010
Last two digits: 1 0


In [2]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee",
     "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge",
     "category": "billing"},
    {"question": "how to reset password",
     "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login",
     "category": "account"},
    {"question": "what are your working hours",
     "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time",
     "category": "general"},
    {"question": "how can i pay the fee",
     "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee",
     "category": "billing"},
]

categories = ["billing", "account", "general"]

category1 = categories[digit1 % 3]
category2 = categories[digit2 % 3]

# Create exactly two personalized entries based on the last two digits.
custom_entries = [
    {
        "question": "How can I update my registered mobile number?",
        "answer": "Go to Profile Settings, choose Update Mobile Number, and verify the new number.",
        "keywords": "mobile update profile",
        "category": category1
    },
    {
        "question": "How can I get a payment receipt for my transaction?",
        "answer": "Open the Payments section and select the transaction to download its receipt.",
        "keywords": "receipt transaction payment",
        "category": category2
    }
]

all_entries = fixed_entries + custom_entries
df_faq = pd.DataFrame(all_entries)

print("Personalized categories:")
print(f"Digit {digit1} -> {category1}")
print(f"Digit {digit2} -> {category2}")

display(df_faq)


Personalized categories:
Digit 1 -> account
Digit 0 -> billing


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,How can I update my registered mobile number?,"Go to Profile Settings, choose Update Mobile N...",mobile update profile,account
5,How can I get a payment receipt for my transac...,Open the Payments section and select the trans...,receipt transaction payment,billing


## Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

def score_query(query_string, df):
    """Return all FAQ entries ranked by TF-IDF cosine similarity."""
    work_df = df.copy()
    work_df["combined_text"] = (
        work_df["question"].fillna("") + " " +
        work_df["keywords"].fillna("")
    )

    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(work_df["combined_text"])
    query_vector = vectorizer.transform([query_string])

    scores = linear_kernel(query_vector, tfidf_matrix).flatten()

    ranked = work_df.copy()
    ranked["confidence_score"] = scores
    ranked = ranked.sort_values("confidence_score", ascending=False)

    return ranked[["question", "answer", "category", "confidence_score"]]

print("--- Q2: Scoring Function ---")
display(score_query("reset my password", df_faq).head())
display(score_query("annual fee payment", df_faq).head())


--- Q2: Scoring Function ---


,question,answer,category,confidence_score
1,how to reset password,Go to Settings > Reset Password.,account,0.942809
0,what is the annual fee,The annual fee is Rs 500.,billing,0.000000
2,what are your working hours,We are open 9 AM to 5 PM.,general,0.000000
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.000000
4,How can I update my registered mobile number?,"Go to Profile Settings, choose Update Mobile N...",account,0.000000


,question,answer,category,confidence_score
0,what is the annual fee,The annual fee is Rs 500.,billing,0.592044
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,0.455563
5,How can I get a payment receipt for my transac...,Open the Payments section and select the trans...,billing,0.268617
1,how to reset password,Go to Settings > Reset Password.,account,0.000000
2,what are your working hours,We are open 9 AM to 5 PM.,general,0.000000


## Q3: Write a function `same_category(category_name, df)` that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [4]:
def same_category(category_name, df):
    """Return all questions belonging to the given category."""
    return df[df["category"] == category_name][["question", "category"]]

# Demonstrate using the categories generated from the roll number.
print(f"Category for digit {digit1}: {category1}")
display(same_category(category1, df_faq))

print(f"Category for digit {digit2}: {category2}")
display(same_category(category2, df_faq))


Category for digit 1: account


,question,category
1,how to reset password,account
4,How can I update my registered mobile number?,account


Category for digit 0: billing


,question,category
0,what is the annual fee,billing
3,how can i pay the fee,billing
5,How can I get a payment receipt for my transac...,billing


## Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named `<your_roll_number>_faq_data.csv`.

In [5]:
# Pick the first FAQ entry and ask the user for a new keyword.
entry_index = 0

print("Selected entry:")
display(df_faq.loc[[entry_index], ["question", "keywords"]])

new_keyword = input("Enter a new keyword to add: ").strip()

if new_keyword:
    df_faq.loc[entry_index, "keywords"] += " " + new_keyword

    output_filename = f"{roll_no}_faq_data.csv"
    df_faq.to_csv(output_filename, index=False)

    print(f"Keyword added: {new_keyword}")
    print(f"Updated file saved as: {output_filename}")
    display(df_faq)
else:
    print("No keyword entered. DataFrame was not changed.")


Selected entry:


,question,keywords
0,what is the annual fee,fee cost price charge


Keyword added: gg
Updated file saved as: 1024160010_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge gg,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,How can I update my registered mobile number?,"Go to Profile Settings, choose Update Mobile N...",mobile update profile,account
5,How can I get a payment receipt for my transac...,Open the Payments section and select the trans...,receipt transaction payment,billing


## Q5: Using `groupby`, print how many FAQ entries you have per category.

In [6]:
print("\n--- Demonstrating Q5: FAQ Entries per Category ---")
category_counts = df_faq.groupby('category').size().reset_index(name='count')
display(category_counts)


--- Demonstrating Q5: FAQ Entries per Category ---


,category,count
0,account,2
1,billing,3
2,general,1


## Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [7]:
def score_query_with_ties(query_string, df):
    """Rank FAQ entries and explicitly display every highest-score tie."""
    ranked = score_query(query_string, df).copy()
    max_score = ranked["confidence_score"].max()

    # np.isclose avoids floating-point equality problems.
    tied = ranked[abs(ranked["confidence_score"] - max_score) < 1e-12]

    if len(tied) > 1:
        print(f"Multiple entries tied for the highest score ({max_score:.4f}):")
        display(tied[["question", "category", "confidence_score"]])
    else:
        print(f"Single highest-scoring entry ({max_score:.4f}):")
        display(tied[["question", "category", "confidence_score"]])

    return ranked

# Non-tie demonstration on the actual knowledge base.
print("--- Q6: Non-tie query ---")
score_query_with_ties("reset password", df_faq)

# Tie demonstration: make two FAQ entries intentionally share the same
# searchable text in a temporary copy, then verify both are returned.
tie_df = df_faq.copy()
tie_df.loc[0, "keywords"] = "tie_demo"
tie_df.loc[3, "keywords"] = "tie_demo"
tie_df.loc[0, "question"] = "tie_demo"
tie_df.loc[3, "question"] = "tie_demo"

print("--- Q6: Tie query ---")
score_query_with_ties("tie_demo", tie_df)


--- Q6: Non-tie query ---
Single highest-scoring entry (0.9428):


,question,category,confidence_score
1,how to reset password,account,0.942809


--- Q6: Tie query ---
Multiple entries tied for the highest score (1.0000):


,question,category,confidence_score
0,tie_demo,billing,1.0
3,tie_demo,billing,1.0


,question,answer,category,confidence_score
0,tie_demo,The annual fee is Rs 500.,billing,1.0
3,tie_demo,"You can pay via UPI, card, or net banking.",billing,1.0
1,how to reset password,Go to Settings > Reset Password.,account,0.0
2,what are your working hours,We are open 9 AM to 5 PM.,general,0.0
4,How can I update my registered mobile number?,"Go to Profile Settings, choose Update Mobile N...",account,0.0
5,How can I get a payment receipt for my transac...,Open the Payments section and select the trans...,billing,0.0
